In [6]:
import os
from dotenv import load_dotenv
from supabase import create_client
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

# Supabase
supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

# Embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [7]:
def insert_document(text: str, metadata: dict = None):
    embedding = embeddings.embed_query(text)

    supabase.table("documents").insert({
        "content": text,
        "metadata": metadata or {},
        "embedding": embedding
    }).execute()

In [8]:
with open("Zomato", "r") as f:
    data = f.read()

In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

doc = Document(page_content=data)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents([doc])

print(len(chunks))

4


In [14]:
for chunk in chunks:
    insert_document(
        text=chunk.page_content,
        metadata={
            "document_title": "Zomato"
        }
    )